# CallWhisper-8k: FLEURS Clean Hindi Control

Purpose: evaluate clean Hindi read speech with the same model families used on GramVaani. This gives the clean-vs-telephone comparison.

Expected manifest:

```text
/content/drive/MyDrive/call-whisper/clean_control/fleurs_hi_50/fleurs_hi_clean_50.csv
```

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/anshulLuhsna/CallWhisper-8k.git"
REPO_DIR = Path("/content/CallWhisper-8k")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/call-whisper")
FLEURS_MANIFEST = DRIVE_PROJECT_DIR / "clean_control/fleurs_hi_50/fleurs_hi_clean_50.csv"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = "src"
print("Repo:", Path.cwd())
print("Drive project dir:", DRIVE_PROJECT_DIR)
print("FLEURS manifest:", FLEURS_MANIFEST)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!python -m pip install -U pip
!python -m pip install -e ".[dev,colab]"
!apt-get update -qq && apt-get install -y -qq ffmpeg


## Verify Manifest

This checks that all 50 WAV paths in the FLEURS manifest exist in Drive.

In [ ]:
import csv

if not FLEURS_MANIFEST.exists():
    raise FileNotFoundError(f"Missing clean-control manifest: {FLEURS_MANIFEST}")

missing = []
rows = []
with FLEURS_MANIFEST.open(encoding="utf-8") as f:
    for row in csv.DictReader(f):
        rows.append(row)
        if not Path(row["audio_path"]).exists():
            missing.append(row["audio_path"])

print("Rows:", len(rows))
print("Missing audio:", len(missing))
if missing[:5]:
    print("Examples:", missing[:5])
assert not missing
rows[0]


## OpenAI Whisper Runs

Run vanilla Whisper `medium` and `large-v3` on clean FLEURS Hindi. Add `small` only if you want direct parity with the earlier local smoke tests.

In [ ]:
openai_runs = [
    (
        "medium",
        str(FLEURS_MANIFEST),
        "results/colab_whisper_medium_fleurs_hi_clean_50_seed0.json",
    ),
    (
        "large-v3",
        str(FLEURS_MANIFEST),
        "results/colab_whisper_large_v3_fleurs_hi_clean_50_seed0.json",
    ),
]

for model, manifest, output_json in openai_runs:
    cmd = [
        sys.executable, "-m", "callwhisper.eval",
        "--manifest", manifest,
        "--model", model,
        "--language-mode", "manifest",
        "--seed", "0",
        "--output-json", output_json,
    ]
    print("RUN", " ".join(cmd))
    subprocess.run(cmd, check=True, env={**os.environ, "PYTHONPATH": "src"})


## Hindi-Tuned Model Run

Run ARTPARK Hindi-tuned Whisper on the same clean-control manifest.

In [ ]:
hf_runs = [
    (
        "ARTPARK-IISc/whisper-medium-vaani-hindi",
        str(FLEURS_MANIFEST),
        "results/colab_hf_artpark_iisc_whisper_medium_vaani_hindi_fleurs_hi_clean_50_seed0.json",
    ),
]

for model_id, manifest, output_json in hf_runs:
    cmd = [
        sys.executable, "-m", "callwhisper.eval.hf_runner",
        "--manifest", manifest,
        "--model-id", model_id,
        "--language-mode", "manifest",
        "--seed", "0",
        "--device", "auto",
        "--output-json", output_json,
    ]
    print("RUN", " ".join(cmd))
    subprocess.run(cmd, check=True, env={**os.environ, "PYTHONPATH": "src"})


## Summarize Clean-Control Results

In [ ]:
import json
import pandas as pd

summary_files = [
    Path("results/colab_whisper_medium_fleurs_hi_clean_50_seed0.json"),
    Path("results/colab_whisper_large_v3_fleurs_hi_clean_50_seed0.json"),
    Path("results/colab_hf_artpark_iisc_whisper_medium_vaani_hindi_fleurs_hi_clean_50_seed0.json"),
]

summary_rows = []
for path in summary_files:
    if not path.exists():
        continue
    payload = json.loads(path.read_text(encoding="utf-8"))
    sample0 = payload["samples"][0]
    summary_rows.append({
        "file": str(path),
        "model": sample0["model"],
        "slice": sample0["slice"],
        "condition": sample0["condition"],
        "files": payload["summary"]["num_files"],
        "wer": round(payload["summary"]["wer"], 4),
        "cer": round(payload["summary"]["cer"], 4),
    })

df = pd.DataFrame(summary_rows).sort_values("wer")
df


In [ ]:
summary_path = Path("results/colab_fleurs_clean_control_summary.md")
summary_path.write_text(df.to_markdown(index=False) + "\n", encoding="utf-8")
print(summary_path.read_text(encoding="utf-8"))


## Save Results To Drive

This copies the FLEURS clean-control JSON and summary files into `MyDrive/call-whisper/results/` so they survive Colab runtime resets.

In [ ]:
import shutil

drive_results = DRIVE_PROJECT_DIR / "results"
drive_results.mkdir(parents=True, exist_ok=True)

for path in Path("results").glob("colab_*fleurs*.json"):
    shutil.copy2(path, drive_results / path.name)
    print("saved", drive_results / path.name)

if summary_path.exists():
    shutil.copy2(summary_path, drive_results / summary_path.name)
    print("saved", drive_results / summary_path.name)
